# Zaid Hybrid IDS — Step 8: SHAP Explanations and Faithfulness
This notebook performs post-hoc SHAP analysis of the frozen final detector. It uses a fixed stratified test sample plus every misclassification, verifies SHAP additivity, exports local and global evidence, and compares top-5 SHAP masking against five repeated random-feature masking controls. No model training, threshold tuning, or detector modification is performed.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q shap scipy

In [3]:
from pathlib import Path
import json
import platform
import sys
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import shap
import sklearn
from scipy.stats import wilcoxon

PROJECT_DIR = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
SPLIT_DIR = PROJECT_DIR / 'data' / 'splits'
RESULT_DIR = PROJECT_DIR / 'results'
MODEL_DIR = PROJECT_DIR / 'models'
FIGURE_DIR = PROJECT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = SPLIT_DIR / 'train.parquet'
TEST_PATH = SPLIT_DIR / 'test.parquet'
FEATURE_MANIFEST_PATH = RESULT_DIR / 'feature_sets_manifest.json'
FINAL_PREDICTIONS_PATH = RESULT_DIR / 'final_frozen_test_predictions.parquet'
FINAL_MANIFEST_PATH = RESULT_DIR / 'final_frozen_test_manifest.json'
MODEL_PATH = MODEL_DIR / 'random_forest_no_dst_port.joblib'
SEED = 20260829
THRESHOLD = 0.5

for path in [TRAIN_PATH, TEST_PATH, FEATURE_MANIFEST_PATH, FINAL_PREDICTIONS_PATH, FINAL_MANIFEST_PATH, MODEL_PATH]:
    assert path.exists(), f'Missing required file: {path}'

with open(FEATURE_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    feature_manifest = json.load(file_handle)
with open(FINAL_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    final_manifest = json.load(file_handle)
FEATURES = feature_manifest['no_dst_port_feature_set']
assert len(FEATURES) == 60 and 'Dst Port' not in FEATURES

print('Loading only the columns required for explanation analysis...')
train_df = pd.read_parquet(TRAIN_PATH, columns=[*FEATURES, 'Label'])
test_df = pd.read_parquet(TEST_PATH, columns=[*FEATURES, 'Label', 'flow_group_id'])
predictions = pd.read_parquet(FINAL_PREDICTIONS_PATH)
frozen_model = joblib.load(MODEL_PATH)
assert len(test_df) == len(predictions) == 156717
assert np.array_equal(test_df['flow_group_id'].to_numpy(), predictions['flow_group_id'].to_numpy())
print('Train rows for baseline statistics:', len(train_df))
print('Test rows for post-hoc analysis:', len(test_df))
print('Model modification in this notebook: False')
print('SHAP environment: READY')

Loading only the columns required for explanation analysis...
Train rows for baseline statistics: 731315
Test rows for post-hoc analysis: 156717
Model modification in this notebook: False
SHAP environment: READY


In [4]:
rng = np.random.default_rng(SEED)
correct = predictions['correct'].astype(bool)
ftp_tp = predictions.index[(predictions['original_label'].eq('FTP-BruteForce')) & correct & predictions['prediction'].eq(1)].to_numpy()
ssh_tp = predictions.index[(predictions['original_label'].eq('SSH-Bruteforce')) & correct & predictions['prediction'].eq(1)].to_numpy()
benign_tn = predictions.index[(predictions['original_label'].eq('Benign')) & correct & predictions['prediction'].eq(0)].to_numpy()
error_rows = predictions.index[~correct].to_numpy()

selected_groups = {
    'ftp_true_positive': np.sort(rng.choice(ftp_tp, size=400, replace=False)),
    'ssh_true_positive': np.sort(rng.choice(ssh_tp, size=400, replace=False)),
    'benign_true_negative': np.sort(rng.choice(benign_tn, size=200, replace=False)),
    'misclassification': np.sort(error_rows)
}
sample_rows = np.concatenate(list(selected_groups.values()))
selection_labels = np.concatenate([
    np.repeat(group_name, len(group_rows))
    for group_name, group_rows in selected_groups.items()
])
assert len(sample_rows) == 1004
assert len(np.unique(sample_rows)) == len(sample_rows)

X_explain = test_df.iloc[sample_rows][FEATURES].astype(np.float32).reset_index(drop=True)
sample_metadata = predictions.iloc[sample_rows].reset_index(drop=True).copy()
sample_metadata['selection_group'] = selection_labels
sample_metadata['source_test_row'] = sample_rows
sample_metadata_path = RESULT_DIR / 'shap_sample_index.csv'
sample_metadata.to_csv(sample_metadata_path, index=False)

print(sample_metadata['selection_group'].value_counts().to_string())
print('Total SHAP sample:', len(sample_metadata))
print('Every final-test error included:', set(error_rows).issubset(set(sample_rows)))
print('Saved:', sample_metadata_path)
print('SHAP sample selection: PASSED')

selection_group
ftp_true_positive       400
ssh_true_positive       400
benign_true_negative    200
misclassification         4
Total SHAP sample: 1004
Every final-test error included: True
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/shap_sample_index.csv
SHAP sample selection: PASSED


In [5]:
print('Computing TreeSHAP values...')
explainer = shap.TreeExplainer(
    frozen_model, feature_perturbation='tree_path_dependent'
)
raw_shap_values = explainer.shap_values(X_explain, check_additivity=False)

if isinstance(raw_shap_values, list):
    attack_shap_values = np.asarray(raw_shap_values[1])
else:
    raw_shap_values = np.asarray(raw_shap_values)
    if raw_shap_values.ndim == 3 and raw_shap_values.shape[-1] == 2:
        attack_shap_values = raw_shap_values[:, :, 1]
    elif raw_shap_values.ndim == 3 and raw_shap_values.shape[0] == 2:
        attack_shap_values = raw_shap_values[1]
    elif raw_shap_values.ndim == 2:
        attack_shap_values = raw_shap_values
    else:
        raise ValueError(f'Unexpected SHAP shape: {raw_shap_values.shape}')

expected_value = np.asarray(explainer.expected_value)
expected_attack_value = float(expected_value[1] if expected_value.ndim else expected_value)
assert attack_shap_values.shape == (len(X_explain), len(FEATURES))
model_attack_probability = frozen_model.predict_proba(X_explain)[:, 1]
reconstructed_probability = expected_attack_value + attack_shap_values.sum(axis=1)
additivity_residual = np.abs(reconstructed_probability - model_attack_probability)

print('SHAP shape:', attack_shap_values.shape)
print('Expected attack value:', expected_attack_value)
print('Mean additivity residual:', float(additivity_residual.mean()))
print('Maximum additivity residual:', float(additivity_residual.max()))
assert float(additivity_residual.max()) < 1e-5
print('SHAP computation and additivity: PASSED')

Computing TreeSHAP values...
SHAP shape: (1004, 60)
Expected attack value: 0.49999999999999634
Mean additivity residual: 4.806652821132781e-13
Maximum additivity residual: 4.815759336446179e-13
SHAP computation and additivity: PASSED


In [6]:
feature_array = np.asarray(FEATURES)
global_importance = pd.DataFrame({
    'feature': FEATURES,
    'mean_abs_shap_all': np.abs(attack_shap_values).mean(axis=0),
    'mean_shap_all': attack_shap_values.mean(axis=0)
})
for group_name in selected_groups:
    group_mask = sample_metadata['selection_group'].eq(group_name).to_numpy()
    global_importance[f'mean_abs_shap_{group_name}'] = np.abs(
        attack_shap_values[group_mask]
    ).mean(axis=0)
global_importance = global_importance.sort_values(
    'mean_abs_shap_all', ascending=False
).reset_index(drop=True)
global_importance['rank'] = np.arange(1, len(global_importance) + 1)

top_feature_records = []
explanation_inputs = []
for sample_index in range(len(X_explain)):
    ordered_indices = np.argsort(np.abs(attack_shap_values[sample_index]))[::-1][:10]
    top_features = []
    for rank, feature_index in enumerate(ordered_indices, start=1):
        record = {
            'sample_index': int(sample_index),
            'source_test_row': int(sample_metadata.loc[sample_index, 'source_test_row']),
            'flow_group_id': int(sample_metadata.loc[sample_index, 'flow_group_id']),
            'selection_group': str(sample_metadata.loc[sample_index, 'selection_group']),
            'original_label': str(sample_metadata.loc[sample_index, 'original_label']),
            'prediction': int(sample_metadata.loc[sample_index, 'prediction']),
            'attack_probability': float(model_attack_probability[sample_index]),
            'rank': rank,
            'feature': str(feature_array[feature_index]),
            'feature_value': float(X_explain.iloc[sample_index, feature_index]),
            'shap_value': float(attack_shap_values[sample_index, feature_index]),
            'abs_shap_value': float(abs(attack_shap_values[sample_index, feature_index]))
        }
        top_feature_records.append(record)
        if rank <= 5:
            top_features.append({
                'name': record['feature'], 'value': record['feature_value'],
                'shap_value': record['shap_value']
            })
    if int(sample_metadata.loc[sample_index, 'prediction']) == 1:
        explanation_inputs.append({
            'event_id': f"test-{int(sample_metadata.loc[sample_index, 'source_test_row']):06d}",
            'predicted_class': 'Attack',
            'confidence': float(model_attack_probability[sample_index]),
            'top_features': top_features
        })

local_top_features = pd.DataFrame(top_feature_records)
GLOBAL_PATH = RESULT_DIR / 'shap_global_importance.csv'
LOCAL_PATH = RESULT_DIR / 'shap_local_top_features.parquet'
VALUES_PATH = RESULT_DIR / 'shap_values_sample.npz'
INPUTS_PATH = RESULT_DIR / 'shap_explanation_inputs.jsonl'
global_importance.to_csv(GLOBAL_PATH, index=False)
local_top_features.to_parquet(LOCAL_PATH, index=False, compression='zstd')
np.savez_compressed(
    VALUES_PATH, shap_values=attack_shap_values.astype(np.float32),
    feature_values=X_explain.to_numpy(dtype=np.float32),
    feature_names=np.asarray(FEATURES), sample_rows=sample_rows,
    expected_attack_value=np.asarray([expected_attack_value])
)
with open(INPUTS_PATH, 'w', encoding='utf-8') as file_handle:
    for record in explanation_inputs:
        file_handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print('Top 15 global SHAP features:')
print(global_importance[['rank', 'feature', 'mean_abs_shap_all']].head(15).to_string(index=False))
print('Alert explanation inputs:', len(explanation_inputs))
print('SHAP evidence exports: PASSED')

Top 15 global SHAP features:
 rank           feature  mean_abs_shap_all
    1  Fwd Seg Size Min           0.120627
    2 Init Fwd Win Byts           0.069936
    3        Bwd Pkts/s           0.043303
    4     Flow Duration           0.036200
    5      Flow IAT Max           0.023704
    6    Bwd Header Len           0.020073
    7     Flow IAT Mean           0.019511
    8       Flow Pkts/s           0.018745
    9 Fwd Act Data Pkts           0.018363
   10    Fwd Header Len           0.018014
   11        Fwd Pkts/s           0.015726
   12      Flow IAT Min           0.015397
   13 Init Bwd Win Byts           0.014577
   14      Fwd IAT Mean           0.012420
   15      Pkt Size Avg           0.009490
Alert explanation inputs: 802
SHAP evidence exports: PASSED


In [7]:
TOP_K = 5
RANDOM_REPEATS = 5
true_positive_mask = (sample_metadata['binary_label'].to_numpy() == 1) & (sample_metadata['prediction'].to_numpy() == 1)
fidelity_sample_indices = np.flatnonzero(true_positive_mask)
X_fidelity = X_explain.iloc[fidelity_sample_indices].copy()
fidelity_shap = attack_shap_values[fidelity_sample_indices]
original_probability = model_attack_probability[fidelity_sample_indices]
benign_baseline = train_df.loc[train_df['Label'].eq('Benign'), FEATURES].median().to_numpy(dtype=np.float32)

top_indices = np.argsort(np.abs(fidelity_shap), axis=1)[:, -TOP_K:]
X_top_masked = X_fidelity.copy()
for row_index in range(len(X_top_masked)):
    selected = top_indices[row_index]
    X_top_masked.iloc[row_index, selected] = benign_baseline[selected]
top_masked_probability = frozen_model.predict_proba(X_top_masked)[:, 1]

random_probability_runs = []
all_feature_indices = np.arange(len(FEATURES))
fidelity_rng = np.random.default_rng(SEED)
for repeat in range(RANDOM_REPEATS):
    X_random_masked = X_fidelity.copy()
    for row_index in range(len(X_random_masked)):
        available = np.setdiff1d(
            all_feature_indices, top_indices[row_index], assume_unique=False
        )
        selected = fidelity_rng.choice(available, size=TOP_K, replace=False)
        X_random_masked.iloc[row_index, selected] = benign_baseline[selected]
    random_probability_runs.append(
        frozen_model.predict_proba(X_random_masked)[:, 1]
    )
random_masked_probability = np.mean(np.vstack(random_probability_runs), axis=0)
top_probability_drop = original_probability - top_masked_probability
random_probability_drop = original_probability - random_masked_probability
paired_difference = top_probability_drop - random_probability_drop
wilcoxon_result = wilcoxon(
    top_probability_drop, random_probability_drop, alternative='greater',
    zero_method='zsplit'
)

fidelity_results = pd.DataFrame({
    'sample_index': fidelity_sample_indices,
    'source_test_row': sample_metadata.iloc[fidelity_sample_indices]['source_test_row'].to_numpy(),
    'original_probability': original_probability,
    'top5_masked_probability': top_masked_probability,
    'random5_masked_probability_mean': random_masked_probability,
    'top5_probability_drop': top_probability_drop,
    'random5_probability_drop_mean': random_probability_drop,
    'paired_drop_difference': paired_difference,
    'top5_decision_flip': top_masked_probability < THRESHOLD,
    'random5_decision_flip': random_masked_probability < THRESHOLD
})
fidelity_summary = {
    'true_positive_events': int(len(fidelity_results)),
    'top_k': TOP_K, 'random_repeats': RANDOM_REPEATS,
    'replacement_baseline': 'training benign median',
    'mean_top5_probability_drop': float(top_probability_drop.mean()),
    'median_top5_probability_drop': float(np.median(top_probability_drop)),
    'mean_random5_probability_drop': float(random_probability_drop.mean()),
    'median_random5_probability_drop': float(np.median(random_probability_drop)),
    'mean_paired_drop_difference': float(paired_difference.mean()),
    'top5_greater_than_random_fraction': float((paired_difference > 0).mean()),
    'top5_decision_flip_rate': float((top_masked_probability < THRESHOLD).mean()),
    'random5_decision_flip_rate': float((random_masked_probability < THRESHOLD).mean()),
    'wilcoxon_statistic': float(wilcoxon_result.statistic),
    'wilcoxon_one_sided_p_value': float(wilcoxon_result.pvalue)
}
FIDELITY_PATH = RESULT_DIR / 'shap_faithfulness_event_results.parquet'
FIDELITY_SUMMARY_PATH = RESULT_DIR / 'shap_faithfulness_summary.json'
fidelity_results.to_parquet(FIDELITY_PATH, index=False, compression='zstd')
with open(FIDELITY_SUMMARY_PATH, 'w', encoding='utf-8') as file_handle:
    json.dump(fidelity_summary, file_handle, ensure_ascii=False, indent=2)
print(pd.Series(fidelity_summary).to_string())
print('SHAP faithfulness comparison: PASSED')

true_positive_events                                    800
top_k                                                     5
random_repeats                                            5
replacement_baseline                 training benign median
mean_top5_probability_drop                         0.813745
median_top5_probability_drop                        0.85738
mean_random5_probability_drop                      0.085809
median_random5_probability_drop                     0.08345
mean_paired_drop_difference                        0.727936
top5_greater_than_random_fraction                       1.0
top5_decision_flip_rate                                 1.0
random5_decision_flip_rate                              0.0
wilcoxon_statistic                                 320400.0
wilcoxon_one_sided_p_value                              0.0
SHAP faithfulness comparison: PASSED


In [8]:
BAR_PATH = FIGURE_DIR / 'shap_global_importance_bar.png'
BEESWARM_PATH = FIGURE_DIR / 'shap_beeswarm.png'
FIDELITY_FIGURE_PATH = FIGURE_DIR / 'shap_faithfulness_comparison.png'

plt.figure()
shap.summary_plot(
    attack_shap_values, X_explain, feature_names=FEATURES,
    plot_type='bar', max_display=15, show=False
)
plt.tight_layout()
plt.savefig(BAR_PATH, dpi=300, bbox_inches='tight')
plt.close()

plt.figure()
shap.summary_plot(
    attack_shap_values, X_explain, feature_names=FEATURES,
    max_display=15, show=False
)
plt.tight_layout()
plt.savefig(BEESWARM_PATH, dpi=300, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(
    [top_probability_drop, random_probability_drop],
    tick_labels=['Top-5 SHAP', 'Random-5 mean'], showfliers=False
)
ax.set_ylabel('Attack-probability drop')
ax.set_title('Faithfulness of SHAP-selected features')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(FIDELITY_FIGURE_PATH, dpi=300, bbox_inches='tight')
plt.close(fig)

manifest = {
    'stage': 'post_hoc_shap_and_faithfulness',
    'seed': SEED, 'threshold': THRESHOLD,
    'model': 'frozen random_forest_no_dst_port',
    'feature_count': len(FEATURES),
    'sample_size': int(len(sample_metadata)),
    'sample_composition': {
        str(key): int(value)
        for key, value in sample_metadata['selection_group'].value_counts().items()
    },
    'all_final_test_errors_included': True,
    'expected_attack_value': expected_attack_value,
    'mean_additivity_residual': float(additivity_residual.mean()),
    'max_additivity_residual': float(additivity_residual.max()),
    'top_10_global_features': global_importance['feature'].head(10).tolist(),
    'faithfulness': fidelity_summary,
    'model_training_or_tuning_performed': False,
    'post_test_use': 'explanation analysis only; detector remains frozen',
    'shap_version': shap.__version__,
    'scipy_version': scipy.__version__,
    'sklearn_version': sklearn.__version__,
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'python_version': sys.version,
    'platform': platform.platform()
}
MANIFEST_PATH = RESULT_DIR / 'shap_analysis_manifest.json'
with open(MANIFEST_PATH, 'w', encoding='utf-8') as file_handle:
    json.dump(manifest, file_handle, ensure_ascii=False, indent=2)

print('Saved:', BAR_PATH)
print('Saved:', BEESWARM_PATH)
print('Saved:', FIDELITY_FIGURE_PATH)
print('Saved:', MANIFEST_PATH)
print('SHAP ANALYSIS PASSED: local evidence, global importance, and faithfulness results saved.')

Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/figures/shap_global_importance_bar.png
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/figures/shap_beeswarm.png
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/figures/shap_faithfulness_comparison.png
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/shap_analysis_manifest.json
SHAP ANALYSIS PASSED: local evidence, global importance, and faithfulness results saved.
